# SWE-Star preview notebook

This notebook loads the public **LogicStar/SWE-Star** dataset from Hugging Face and previews the first examples.

**Source**  
The dataset card shows one subset (`default`) and one split (`train`) with about **244k rows**. The visible fields include `instance_id`, `exit_status`, `stitched`, `full`, `result`, and `resolved`. citeturn513868view0turn513868view1

In [ ]:
!pip -q install datasets pandas

In [ ]:
import json
import ast
from pathlib import Path
from typing import Any
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_colwidth", 200)

def truncate(text: Any, n: int = 300) -> str:
    if text is None:
        return ""
    s = str(text)
    return s if len(s) <= n else s[:n] + " …"

def maybe_parse_json(x: Any):
    if isinstance(x, (dict, list)):
        return x
    if not isinstance(x, str):
        return x
    x = x.strip()
    if not x:
        return x
    for fn in (json.loads, ast.literal_eval):
        try:
            return fn(x)
        except Exception:
            pass
    return x

def first_dialogue_excerpt(obj: Any, max_items: int = 4) -> str:
    parsed = maybe_parse_json(obj)
    if isinstance(parsed, list):
        parts = []
        for item in parsed[:max_items]:
            if isinstance(item, dict):
                role = item.get("role") or item.get("userType") or item.get("type") or "item"
                content = item.get("content") or item.get("text") or item.get("message") or item.get("action") or item.get("thought") or item
                parts.append(f"{role}: {truncate(content, 180)}")
            else:
                parts.append(truncate(item, 180))
        return "\n".join(parts)
    if isinstance(parsed, dict):
        return truncate(json.dumps(parsed, indent=2), 500)
    return truncate(parsed, 500)

def choose_first_existing(columns, candidates):
    for c in candidates:
        if c in columns:
            return c
    return None

In [ ]:
from datasets import load_dataset

ds = load_dataset("LogicStar/SWE-Star", split="train")
df = ds.to_pandas()
print(df.shape)
display(df.head())

In [ ]:
cols = [c for c in ["timestamp", "instance_id", "exit_status", "resolved"] if c in df.columns]
display(df[cols].head(10))

In [ ]:
trajectory_col = choose_first_existing(df.columns, ["stitched", "full"])
examples = []
for _, row in df.head(5).iterrows():
    examples.append({
        "instance_id": row.get("instance_id"),
        "exit_status": row.get("exit_status"),
        "resolved": row.get("resolved"),
        "trajectory_excerpt": first_dialogue_excerpt(row.get(trajectory_col)),
        "result_excerpt": truncate(row.get("result"), 250),
    })
example_df = pd.DataFrame(examples)
display(example_df)

In [ ]:
if "exit_status" in df.columns:
    display(df["exit_status"].value_counts().rename_axis("exit_status").reset_index(name="count"))
if "resolved" in df.columns:
    display(df["resolved"].value_counts(dropna=False).rename_axis("resolved").reset_index(name="count"))